# YOLOv8m - Emergency Vehicle Detection
**Proyecto:** Getaway Car Identification and Tracking in Drone Surveillance  
**Training hardware:** Kaggle 2x T4 GPUs  
**Inferencia:** Jetson Orin Nano  
**Clases:** ambulance, firetruck, police

## 1. Instalacion de dependencias

In [ ]:
!pip install ultralytics roboflow albumentations pyyaml opencv-python-headless -q

## 2. Imports

In [ ]:
import os
import shutil
import random
import yaml
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau

from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import albumentations as A

random.seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 3. Descarga del dataset desde Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="<YOUR_API_KEY>")
project = rf.workspace("luiss-workspace-wchns").project("emergency-vehicles-merged-v2")
version = project.version(1)
dataset = version.download("yolov8")

DATASET_PATH = Path(dataset.location)
print(f"Dataset descargado en: {DATASET_PATH}")
print("Estructura:")
for p in sorted(DATASET_PATH.rglob("*")):
    if p.is_dir():
        print(f"  {p.relative_to(DATASET_PATH)}/")

## 4. Analisis de estructura del dataset

In [ ]:
# Leer data.yaml
yaml_path = DATASET_PATH / "data.yaml"
with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)

CLASS_NAMES = data_cfg["names"]
NUM_CLASSES = data_cfg["nc"]
print(f"Clases ({NUM_CLASSES}): {CLASS_NAMES}")

# Contar imagenes por split
for split in ["train", "valid", "test"]:
    img_dir = DATASET_PATH / split / "images"
    if img_dir.exists():
        imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpeg"))
        print(f"  {split}: {len(imgs)} imagenes")

## 5. Analisis de desbalance de clases

In [ ]:
def analyze_dataset(split_dir: Path, class_names: list):
    """Analiza la distribucion de clases en un split del dataset.

    Retorna:
        instance_counts: Counter con instancias (anotaciones) por clase
        images_per_class: dict {class_id: [lista de rutas de imagenes]}
        background_images: lista de imagenes sin anotaciones
    """
    images_dir = split_dir / "images"
    labels_dir = split_dir / "labels"

    image_files = (
        list(images_dir.glob("*.jpg"))
        + list(images_dir.glob("*.png"))
        + list(images_dir.glob("*.jpeg"))
    )

    instance_counts = Counter()
    images_per_class = defaultdict(list)
    background_images = []

    for img_path in image_files:
        label_path = labels_dir / (img_path.stem + ".txt")

        if not label_path.exists():
            background_images.append(img_path)
            continue

        with open(label_path) as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]

        if not lines:
            background_images.append(img_path)
            continue

        classes_in_image = set()
        for line in lines:
            cls_id = int(float(line.split()[0]))
            instance_counts[cls_id] += 1
            classes_in_image.add(cls_id)

        for cls_id in classes_in_image:
            images_per_class[cls_id].append(img_path)

    return instance_counts, images_per_class, background_images


train_dir = DATASET_PATH / "train"
instance_counts, images_per_class, background_images = analyze_dataset(train_dir, CLASS_NAMES)

print("=== Distribucion del set de entrenamiento ===")
print(f"\nInstancias (anotaciones) por clase:")
for cls_id, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {instance_counts[cls_id]}")

print(f"\nImagenes que contienen cada clase:")
for cls_id, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {len(images_per_class[cls_id])} imagenes")

print(f"\nImagenes background (sin anotaciones): {len(background_images)}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = CLASS_NAMES
img_counts = [len(images_per_class[i]) for i in range(len(CLASS_NAMES))]
inst_counts = [instance_counts[i] for i in range(len(CLASS_NAMES))]

colors = ["#2196F3", "#FF5722", "#4CAF50"]

axes[0].bar(names, img_counts, color=colors)
axes[0].set_title("Imagenes por clase (train)")
axes[0].set_ylabel("# Imagenes")
for i, v in enumerate(img_counts):
    axes[0].text(i, v + 0.5, str(v), ha="center", fontweight="bold")

axes[1].bar(names, inst_counts, color=colors)
axes[1].set_title("Instancias por clase (train)")
axes[1].set_ylabel("# Instancias")
for i, v in enumerate(inst_counts):
    axes[1].text(i, v + 0.5, str(v), ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("/kaggle/working/class_distribution_before.png", dpi=150)
plt.show()
print("Plot guardado.")

## 6. Balanceo de clases por oversampling con augmentaciones offline

In [ ]:
# Pipeline de augmentacion compatible con YOLO bboxes
aug_transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=35, val_shift_limit=25, p=0.5),
        A.Rotate(limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
        A.RandomScale(scale_limit=0.15, p=0.4),
        A.GaussNoise(p=0.3),
        A.MotionBlur(blur_limit=5, p=0.2),
        A.RandomShadow(p=0.2),
    ],
    bbox_params=A.BboxParams(
        format="yolo",
        label_fields=["class_labels"],
        min_visibility=0.3,
    ),
)


def read_yolo_label(label_path: Path):
    """Lee un archivo de label YOLO. Retorna lista de (cls_id, cx, cy, w, h)."""
    if not label_path.exists():
        return []
    rows = []
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                cls_id = int(float(parts[0]))
                cx, cy, w, h = map(float, parts[1:])
                rows.append((cls_id, cx, cy, w, h))
    return rows


def write_yolo_label(label_path: Path, rows):
    """Escribe un archivo de label YOLO."""
    with open(label_path, "w") as f:
        for cls_id, cx, cy, w, h in rows:
            f.write(f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")


def augment_and_save(src_img_path: Path, src_label_path: Path,
                     dst_img_path: Path, dst_label_path: Path,
                     transform):
    """Aplica augmentacion a una imagen+label y guarda el resultado."""
    img = cv2.imread(str(src_img_path))
    if img is None:
        return False
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    rows = read_yolo_label(src_label_path)

    bboxes = [(cx, cy, w, h) for _, cx, cy, w, h in rows]
    class_labels = [cls_id for cls_id, *_ in rows]

    # Clamp bboxes para evitar errores de Albumentations
    clamped_bboxes = []
    for (cx, cy, w, h) in bboxes:
        cx = max(0.0, min(1.0, cx))
        cy = max(0.0, min(1.0, cy))
        w = max(0.001, min(1.0, w))
        h = max(0.001, min(1.0, h))
        clamped_bboxes.append((cx, cy, w, h))

    try:
        result = transform(
            image=img_rgb,
            bboxes=clamped_bboxes,
            class_labels=class_labels,
        )
    except Exception:
        return False

    if not result["bboxes"]:
        return False

    aug_img = cv2.cvtColor(result["image"], cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(dst_img_path), aug_img)

    new_rows = [
        (cls_id, cx, cy, w, h)
        for cls_id, (cx, cy, w, h) in zip(
            result["class_labels"], result["bboxes"]
        )
    ]
    write_yolo_label(dst_label_path, new_rows)
    return True


def balance_dataset(train_dir: Path, images_per_class: dict,
                    class_names: list, transform):
    """Oversamplea clases minoritarias con augmentaciones hasta igualar la clase mayoritaria."""
    images_dir = train_dir / "images"
    labels_dir = train_dir / "labels"

    class_img_counts = {cls_id: len(imgs) for cls_id, imgs in images_per_class.items()}
    target_count = max(class_img_counts.values())
    majority_class = max(class_img_counts, key=class_img_counts.get)

    print(f"Clase mayoritaria: {class_names[majority_class]} ({target_count} imagenes)")
    print(f"Target por clase: {target_count} imagenes\n")

    total_created = 0

    for cls_id, cls_name in enumerate(class_names):
        current_count = class_img_counts.get(cls_id, 0)
        deficit = target_count - current_count

        if deficit <= 0:
            print(f"  {cls_name}: ya balanceado ({current_count} imagenes), sin oversampling.")
            continue

        print(f"  {cls_name}: {current_count} imagenes -> necesita {deficit} augmentadas.")

        source_images = images_per_class[cls_id]
        created = 0
        attempts = 0
        max_attempts = deficit * 3

        while created < deficit and attempts < max_attempts:
            src_img = random.choice(source_images)
            src_label = labels_dir / (src_img.stem + ".txt")

            suffix = f"_aug_{cls_name}_{created:04d}"
            ext = src_img.suffix
            dst_img = images_dir / (src_img.stem + suffix + ext)
            dst_label = labels_dir / (src_img.stem + suffix + ".txt")

            success = augment_and_save(src_img, src_label, dst_img, dst_label, transform)
            if success:
                created += 1
            attempts += 1

        print(f"    Creadas: {created} imagenes augmentadas.")
        total_created += created

    print(f"\nTotal imagenes augmentadas creadas: {total_created}")
    return total_created


print("Balanceando dataset de entrenamiento...")
total_aug = balance_dataset(train_dir, images_per_class, CLASS_NAMES, aug_transform)
print("\nBalanceo completado.")

## 7. Verificacion del balance y manejo de imagenes background

In [ ]:
# Asegurar que todas las imagenes background tienen un .txt vacio
labels_dir = train_dir / "labels"
labels_dir.mkdir(exist_ok=True)

fixed_bg = 0
for img_path in background_images:
    label_path = labels_dir / (img_path.stem + ".txt")
    if not label_path.exists():
        label_path.touch()
        fixed_bg += 1

print(f"Labels vacios creados para {fixed_bg} imagenes background adicionales.")
print(f"Total imagenes background en train: {len(background_images)}")
print("(Las imagenes sin anotaciones ensenyan al modelo a no detectar cuando no hay vehiculos de emergencia.)")

# Re-analizar para verificar el balance
new_counts, new_imgs_per_class, _ = analyze_dataset(train_dir, CLASS_NAMES)

print("\n=== Distribucion DESPUES del balanceo ===")
for cls_id, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {len(new_imgs_per_class[cls_id])} imagenes, {new_counts[cls_id]} instancias")

# Plot comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

before_counts = [len(images_per_class[i]) for i in range(len(CLASS_NAMES))]
after_counts = [len(new_imgs_per_class[i]) for i in range(len(CLASS_NAMES))]

x = np.arange(len(CLASS_NAMES))
width = 0.35

axes[0].bar(x - width/2, before_counts, width, label="Antes", color="#FF7043")
axes[0].bar(x + width/2, after_counts, width, label="Despues", color="#42A5F5")
axes[0].set_xticks(x)
axes[0].set_xticklabels(CLASS_NAMES)
axes[0].set_title("Imagenes por clase: antes vs despues del balanceo")
axes[0].set_ylabel("# Imagenes")
axes[0].legend()

# Proporcion relativa
total_after = sum(after_counts)
proportions = [c / total_after * 100 for c in after_counts]
axes[1].pie(proportions, labels=CLASS_NAMES, autopct="%1.1f%%",
            colors=["#2196F3", "#FF5722", "#4CAF50"])
axes[1].set_title("Proporcion de clases (despues del balanceo)")

plt.tight_layout()
plt.savefig("/kaggle/working/class_distribution_after.png", dpi=150)
plt.show()

## 8. Generacion del data.yaml balanceado

In [ ]:
balanced_yaml_path = DATASET_PATH / "balanced_data.yaml"

balanced_cfg = {
    "path": str(DATASET_PATH),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": NUM_CLASSES,
    "names": CLASS_NAMES,
}

with open(balanced_yaml_path, "w") as f:
    yaml.dump(balanced_cfg, f, default_flow_style=False, allow_unicode=True)

print(f"data.yaml balanceado guardado en: {balanced_yaml_path}")
print("Contenido:")
with open(balanced_yaml_path) as f:
    print(f.read())

## 9. PlateauDetectionTrainer — Scheduler personalizado para DDP

In [ ]:

trainer_code = '''
import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau
from ultralytics.models.yolo.detect import DetectionTrainer

class _PlateauWrapper:
    def __init__(self, optimizer):
        self._scheduler = ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=7,
            threshold=1e-4,
            min_lr=1e-5,
        )
        self._last_metric = float("inf")

    def step(self, metrics=None):
        if metrics is not None:
            self._last_metric = metrics
        self._scheduler.step(self._last_metric)

    def state_dict(self):
        return self._scheduler.state_dict()

    def load_state_dict(self, state_dict):
        return self._scheduler.load_state_dict(state_dict)

    def get_last_lr(self):
        return [pg["lr"] for pg in self._scheduler.optimizer.param_groups]


class PlateauDetectionTrainer(DetectionTrainer):
    def _setup_scheduler(self):
        for pg in self.optimizer.param_groups:
            pg.setdefault("initial_lr", pg["lr"])

        self.lf = lambda x: 1.0

        self.scheduler = _PlateauWrapper(self.optimizer)
        self._plateau_scheduler = True

    def scheduler_step(self):
        if not hasattr(self, "_plateau_scheduler"):
            super().scheduler_step()
            return

        metrics = self.metrics if hasattr(self, "metrics") and self.metrics else {}
        val_loss = (
            metrics.get("val/box_loss", 0.0)
            + metrics.get("val/cls_loss", 0.0)
            + metrics.get("val/dfl_loss", 0.0)
        )
        if val_loss == 0.0:
            val_loss = self.loss.item() if hasattr(self, "loss") and self.loss is not None else float("inf")

        self.scheduler.step(val_loss)

        if self.rank in (-1, 0):
            self.lr = {f"lr/pg{i}": pg["lr"] for i, pg in enumerate(self.optimizer.param_groups)}
'''

import shutil

with open("/kaggle/working/plateau_trainer.py", "w") as f:
    f.write(trainer_code)

shutil.copy(
    "/kaggle/working/plateau_trainer.py",
    "/usr/local/lib/python3.12/dist-packages/plateau_trainer.py"
)

import sys
sys.path.insert(0, "/kaggle/working")
from plateau_trainer import PlateauDetectionTrainer

print("PlateauDetectionTrainer listo.")

## 10. Configuracion y lanzamiento del entrenamiento

In [ ]:
project_dir = Path("/kaggle/working/runs")
project_dir.mkdir(parents=True, exist_ok=True)
run_name = "emergency_vehicles_yolov8m_v1"

# cargar yolo
model = YOLO("yolov8m.pt")

print(f"Modelo: YOLOv8m")
print(f"Dataset: {balanced_yaml_path}")
print(f"Clases: {CLASS_NAMES}")
print(f"Dispositivos: 2x T4 (device=[0, 1])")
print(f"Output: {project_dir / run_name}")

In [ ]:
results = model.train(
    data=str(balanced_yaml_path),
    epochs=100,
    imgsz=640,
    batch=32,
    device=[0, 1],

    # early stopping
    patience=0,

    save=True,
    project=str(project_dir),
    name=run_name,

    # plateau scheduler
    trainer=PlateauDetectionTrainer,
    cos_lr=False,
    lr0=0.01,
    lrf=0.01,
    warmup_epochs=3,

    # onthefly augments
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.15,
    scale=0.5,
    fliplr=0.5,
    mixup=0.15,
    erasing=0.4,

    # apagar augments
    mosaic=0.0,
    close_mosaic=0,
    copy_paste=0.0,
    cutmix=0.0,

    # Loss weights
    cls=1.0,
    box=7.5,
)

print("\nEntrenamiento completado.")
print(f"Mejor modelo: {project_dir / run_name / 'weights/best.pt'}")

## 11. Evaluacion — metricas en test set

In [ ]:
best_model_path = project_dir / run_name / "weights" / "best.pt"
eval_model = YOLO(str(best_model_path))

print("=== Evaluacion en validation set ===")
val_results = eval_model.val(
    data=str(balanced_yaml_path),
    split="val",
    imgsz=640,
    device=0,
)

print("\n=== Evaluacion en test set ===")
test_results = eval_model.val(
    data=str(balanced_yaml_path),
    split="test",
    imgsz=640,
    device=0,
)

# Mostrar metricas por clase
print("\n--- Metricas por clase (test set) ---")
print(f"{'Clase':<15} {'Precision':>10} {'Recall':>10} {'mAP@50':>10} {'mAP@50-95':>12}")
print("-" * 60)

if hasattr(test_results, 'box'):
    box = test_results.box
    for i, name in enumerate(CLASS_NAMES):
        try:
            p = box.p[i] if hasattr(box, 'p') else 0
            r = box.r[i] if hasattr(box, 'r') else 0
            ap50 = box.ap50[i] if hasattr(box, 'ap50') else 0
            ap = box.ap[i] if hasattr(box, 'ap') else 0
            print(f"{name:<15} {p:>10.4f} {r:>10.4f} {ap50:>10.4f} {ap:>12.4f}")
        except Exception:
            pass
    print("-" * 60)
    try:
        print(f"{'MEDIA':<15} {box.mp:>10.4f} {box.mr:>10.4f} {box.map50:>10.4f} {box.map:>12.4f}")
    except Exception:
        pass

# Verificacion del criterio de aceptacion (>=70% mAP@50)
try:
    map50 = test_results.box.map50
    criterio = "CUMPLIDO" if map50 >= 0.70 else "NO CUMPLIDO"
    print(f"\nCriterio de aceptacion (mAP@50 >= 0.70): {map50:.4f} -> {criterio}")
except Exception:
    print("\nNo se pudo obtener mAP@50 automaticamente. Revisar logs.")

## 12. Curvas de entrenamiento

In [ ]:
import pandas as pd

results_csv = project_dir / run_name / "results.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    metrics_to_plot = [
        ("train/box_loss", "Train Box Loss"),
        ("train/cls_loss", "Train Cls Loss"),
        ("val/box_loss", "Val Box Loss"),
        ("val/cls_loss", "Val Cls Loss"),
        ("metrics/mAP50(B)", "mAP@50"),
        ("metrics/mAP50-95(B)", "mAP@50-95"),
    ]

    for ax, (col, title) in zip(axes.flat, metrics_to_plot):
        if col in df.columns:
            ax.plot(df["epoch"], df[col], linewidth=2)
            ax.set_title(title)
            ax.set_xlabel("Epoch")
            ax.grid(True, alpha=0.3)
        else:
            ax.set_title(f"{title} (no disponible)")

    plt.suptitle("Curvas de entrenamiento — YOLOv8m Emergency Vehicles", fontsize=14)
    plt.tight_layout()
    plt.savefig("/kaggle/working/training_curves.png", dpi=150)
    plt.show()
else:
    print("results.csv no encontrado. Verificar ruta del run.")

## 13. Visualizacion de predicciones

In [ ]:
test_images_dir = DATASET_PATH / "test" / "images"
test_images = (
    list(test_images_dir.glob("*.jpg"))
    + list(test_images_dir.glob("*.png"))
    + list(test_images_dir.glob("*.jpeg"))
)

# Seleccionar 8 imagenes aleatorias
sample_images = random.sample(test_images, min(8, len(test_images)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
colors_map = {0: (255, 100, 100), 1: (100, 255, 100), 2: (100, 100, 255)}

for ax, img_path in zip(axes.flat, sample_images):
    pred = eval_model.predict(str(img_path), conf=0.25, verbose=False)[0]
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    for box in pred.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        color = colors_map.get(cls_id, (255, 255, 0))
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
        label = f"{CLASS_NAMES[cls_id]} {conf:.2f}"
        cv2.putText(img_rgb, label, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    ax.imshow(img_rgb)
    ax.set_title(img_path.name[:30], fontsize=8)
    ax.axis("off")

plt.suptitle("Predicciones (conf >= 0.25) — YOLOv8m Emergency Vehicles", fontsize=13)
plt.tight_layout()
plt.savefig("/kaggle/working/predictions_sample.png", dpi=150)
plt.show()

## 14. Inferencia en imagenes background (verificacion de falsos positivos)

In [ ]:
# Buscar imagenes background en test set
test_labels_dir = DATASET_PATH / "test" / "labels"
test_bg_images = []

for img_path in test_images:
    lbl = test_labels_dir / (img_path.stem + ".txt")
    if not lbl.exists() or lbl.stat().st_size == 0:
        test_bg_images.append(img_path)

print(f"Imagenes background en test set: {len(test_bg_images)}")

if test_bg_images:
    sample_bg = random.sample(test_bg_images, min(4, len(test_bg_images)))
    fig, axes = plt.subplots(1, len(sample_bg), figsize=(5 * len(sample_bg), 5))
    if len(sample_bg) == 1:
        axes = [axes]

    false_positives = 0
    for ax, img_path in zip(axes, sample_bg):
        pred = eval_model.predict(str(img_path), conf=0.25, verbose=False)[0]
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        n_det = len(pred.boxes)
        if n_det > 0:
            false_positives += 1
        ax.imshow(img_rgb)
        ax.set_title(f"Detecciones: {n_det} (esperado: 0)", fontsize=9,
                     color="red" if n_det > 0 else "green")
        ax.axis("off")

    print(f"Falsos positivos en muestra: {false_positives}/{len(sample_bg)}")
    plt.suptitle("Imagenes background — verificacion de falsos positivos")
    plt.tight_layout()
    plt.savefig("/kaggle/working/background_check.png", dpi=150)
    plt.show()
else:
    print("No se encontraron imagenes background en el test set.")

## 15. Export del modelo para Jetson Orin Nano

In [ ]:
export_dir = Path("/kaggle/working/exports")
export_dir.mkdir(exist_ok=True)

# 1. Export a ONNX (portable, se puede convertir en la Jetson)
print("Exportando a ONNX...")
try:
    onnx_path = eval_model.export(
        format="onnx",
        imgsz=640,
        half=False,
        simplify=True,
        opset=17,
        dynamic=False,
    )
    shutil.copy(onnx_path, export_dir / "best.onnx")
    print(f"ONNX guardado en: {export_dir / 'best.onnx'}")
except Exception as e:
    print(f"Error al exportar ONNX: {e}")

# 2. Intento de export a TensorRT FP16 (si TensorRT esta disponible en Kaggle)
print("\nIntentando exportar a TensorRT FP16...")
try:
    trt_path = eval_model.export(
        format="engine",
        imgsz=640,
        half=True,
        device=0,
        workspace=4,  # GB
    )
    shutil.copy(trt_path, export_dir / "best_fp16.engine")
    print(f"TensorRT engine guardado en: {export_dir / 'best_fp16.engine'}")
except Exception as e:
    print(f"TensorRT no disponible en este entorno: {e}")
    print()
    print("Para convertir en la Jetson Orin Nano, ejecutar:")
    print("  yolo export model=best.pt format=engine half=True imgsz=640 device=0")
    print()
    print("Alternativa si la latencia supera 200ms:")
    print("  Reducir imgsz a 480 o usar YOLOv8s/YOLOv8n")

## 16. Guardar artefactos finales

In [ ]:
artifacts_dir = Path("/kaggle/working/artifacts")
artifacts_dir.mkdir(exist_ok=True)

# Copiar modelos
run_weights = project_dir / run_name / "weights"
for weight_file in ["best.pt", "last.pt"]:
    src = run_weights / weight_file
    if src.exists():
        shutil.copy(src, artifacts_dir / weight_file)
        print(f"Copiado: {weight_file}")

# Copiar metricas
for f in ["results.csv", "confusion_matrix.png", "results.png"]:
    src = project_dir / run_name / f
    if src.exists():
        shutil.copy(src, artifacts_dir / f)
        print(f"Copiado: {f}")

# Copiar plots generados
for plot in [
    "class_distribution_before.png",
    "class_distribution_after.png",
    "training_curves.png",
    "predictions_sample.png",
    "background_check.png",
]:
    src = Path("/kaggle/working") / plot
    if src.exists():
        shutil.copy(src, artifacts_dir / plot)
        print(f"Copiado: {plot}")

# Copiar data.yaml balanceado
shutil.copy(balanced_yaml_path, artifacts_dir / "balanced_data.yaml")

print("\n=== Artefactos finales ===")
for f in sorted(artifacts_dir.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:<35} {size_mb:.2f} MB")